# Research Version 3 ? 06: Classical source/event transfer

Run 20 fixed-control source/event stress models from scratch. Do not use pooled-domain candidate winners for held-out-domain tuning. Report class support and source differences alongside scores.

This notebook runs only in Colab and Google Drive. No Windows checkout, local datasets, local model files or replication-manifest file are required. Use the new experiment ID; do not mix this package with an older cpu-v3 archive.


## Scope and compatibility patch

Classical domain evaluation plus registration of shared neural infrastructure for 07-09. Registration does not train neural models.


# 06: Twenty source/event-held-out classical experiments

## 1. Purpose

Fit twenty source/event-held-out TF-IDF/LR models under the fixed development-only protocol.

## 2. Prerequisites and outputs

**Required inputs:** Notebook 04 complete baseline summary and shared text helpers.

**This stage saves:** Six source-transfer folds, fourteen event folds, protocol snapshot, pooled metrics and report.

Required earlier notebook markers: 04.

All runtime files belong to this Colab/Drive experiment; no external project files are read.

## 3. Configuration

Use the same experiment ID/folder throughout. All action switches default False. Enable Drive/bootstrap and RUN_STAGE deliberately; enable installation on a fresh runtime. Notebook 01 needs dataset download permission; notebook 08 needs pretrained download permission.

In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


## 4. Readable stage implementation

The small cell command below registers visible module text in memory. It does not execute the module. This notebook supplies only the files listed here; other supporting modules are restored from earlier Colab stages.

In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### configs/generalization.json


In [ ]:
%%chainshield_file configs/generalization.json
{
  "schema_version": 1,
  "protocol_id": "generalization-v1",
  "seed": 42,
  "threads": 2,
  "eligible_original_partitions": [
    "train",
    "dev"
  ],
  "pool_policy": "Eligible original train and development records only; final test excluded",
  "hyperparameter_source": "configs/training.json",
  "hyperparameter_policy": "Frozen TF-IDF/logistic settings; no tuning against holdout results",
  "source_transfer": {
    "dataset": "india_nepal",
    "tasks": [
      "disaster_relevance",
      "disaster_type",
      "humanitarian_information"
    ],
    "directions": [
      {
        "training_domains": [
          "original"
        ],
        "heldout_domain": "india"
      },
      {
        "training_domains": [
          "india"
        ],
        "heldout_domain": "original"
      }
    ]
  },
  "event_holdout": {
    "dataset": "crisismmd",
    "view": "shared",
    "tasks": [
      "informative",
      "humanitarian"
    ],
    "events": [
      "california_wildfires",
      "hurricane_harvey",
      "hurricane_irma",
      "hurricane_maria",
      "iraq_iran_earthquake",
      "mexico_earthquake",
      "srilanka_floods"
    ]
  },
  "isolation_policy": "Remove training records sharing an exact normalized text key or connected group with held-out records",
  "template_policy": "Do not automatically remove mechanical template or semantic candidates",
  "metric_policy": "Report global-class and supported-class macro-F1 plus per-class support",
  "interpretation": "Development stress validation, not an untouched final test; original development data informed baseline comparisons",
  "expected_fold_count": 20,
  "training_config_sha256": "657bcfd9d2363507c0d155cfdb4a6b74e7e1f287d33e5d4874c02484e15702b9"
}


### src/chainshield/nlp/generalization.py


In [ ]:
%%chainshield_file src/chainshield/nlp/generalization.py
"""Source/event holdout helpers with explicit development-only isolation."""
from __future__ import annotations
from collections import Counter
import csv
import hashlib
import io
from pathlib import Path
import numpy as np
from sklearn.metrics import f1_score
from chainshield.data.text_utils import text_key
from chainshield.nlp.baseline import load_text_partitions
from chainshield.ml.train import evaluate_predictions


def validate_pool(pool):
    if not pool or len({r["record_id"] for r in pool})!=len(pool):
        raise ValueError("Expected a nonempty pool with unique record IDs")
    for row in pool:
        if row["original_partition"] not in {"train","dev"}:
            raise ValueError("Final-test or unknown partition forbidden")
        if any(not row.get(key) for key in ("text","text_key","group_id","label","domain")):
            raise ValueError("Incomplete development record")


def load_pool(directory:Path,dataset,task,view=None):
    partitions=load_text_partitions(directory,dataset,task,view)
    selected={}
    for split,part in partitions.items():
        for identifier,text,label,group in zip(part["ids"],part["texts"],part["labels"],part["groups"]):
            if identifier in selected:
                raise ValueError("Duplicate pooled identity")
            selected[identifier]={"record_id":identifier,"text":text,"text_key":text_key(text),
                "label":label,"group_id":group,"original_partition":split}
    identity="tweet_id" if dataset=="crisismmd" else "record_id"
    domain="event_name" if dataset=="crisismmd" else "collection"
    with (directory/"canonical.csv").open(encoding="utf-8",newline="") as handle:
        for row in csv.DictReader(handle):
            if row[identity] in selected:
                selected[row[identity]]["domain"]=row[domain]
    pool=sorted(selected.values(),key=lambda row:row["record_id"])
    validate_pool(pool)
    return pool


def construct_fold(pool,training_domains,heldout_domain,classes):
    validate_pool(pool)
    if not training_domains or heldout_domain in training_domains:
        raise ValueError("Training/held-out domains must be disjoint")
    heldout=[r for r in pool if r["domain"]==heldout_domain]
    if not heldout:
        raise ValueError("Empty held-out domain")
    groups={r["group_id"] for r in heldout}
    texts={r["text_key"] for r in heldout}
    candidates=[r for r in pool if r["domain"] in training_domains]
    train=[r for r in candidates if r["group_id"] not in groups and r["text_key"] not in texts]
    kept={r["record_id"] for r in train}
    purged=[r for r in candidates if r["record_id"] not in kept]
    if not train or set(r["label"] for r in train)!=set(classes):
        raise ValueError("Training fold lacks exactly the declared task labels")
    if not {r["label"] for r in heldout}<=set(classes):
        raise ValueError("Unsupported held-out label")
    roles={r["record_id"]:"training" for r in train}
    roles.update({r["record_id"]:"heldout_development" for r in heldout})
    roles.update({r["record_id"]:"purged_training_overlap" for r in purged})
    manifest=[{key:r[key] for key in ("record_id","original_partition","domain",
                                    "group_id","text_key","label")}
              | {"role":roles.get(r["record_id"],"outside_fold")} for r in pool]
    return {"training":train,"heldout":heldout,"manifest":manifest,
        "training_domains":list(training_domains),"heldout_domain":heldout_domain,
        "purged_training_rows":len(purged),
        "training_label_counts":dict(Counter(r["label"] for r in train)),
        "heldout_label_counts":dict(Counter(r["label"] for r in heldout)),
        "invariants":{"final_test_excluded":True,"domains_disjoint":True,
                      "exact_text_disjoint":True,"connected_groups_disjoint":True}}


def manifest_bytes(rows):
    if not rows:
        raise ValueError("Empty fold manifest")
    handle=io.StringIO(newline="")
    writer=csv.DictWriter(handle,fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    return handle.getvalue().encode("utf-8")


def manifest_digest(rows):
    return hashlib.sha256(manifest_bytes(rows)).hexdigest()


def holdout_metrics(labels,probabilities,classes,positive_label=None):
    metrics=evaluate_predictions(labels,probabilities,classes,positive_label)
    predicted=np.asarray(classes)[np.asarray(probabilities).argmax(axis=1)]
    metrics["macro_f1_supported_classes"]=float(f1_score(labels,predicted,
        labels=sorted(set(labels)),average="macro",zero_division=0))
    metrics["evaluation_design"]="held-out source/event within original train/development pool"
    metrics["test_evaluated"]=False
    return metrics


def aggregate_predictions(folds,expected_ids,classes,positive_label=None):
    ids=[identifier for fold in folds for identifier in fold["ids"]]
    if len(ids)!=len(set(ids)) or set(ids)!=set(expected_ids):
        raise ValueError("Holdout predictions must cover every pooled record exactly once")
    if any(list(fold["classes"])!=list(classes) for fold in folds):
        raise ValueError("Inconsistent probability column order")
    labels=[label for fold in folds for label in fold["labels"]]
    probabilities=np.vstack([fold["probabilities"] for fold in folds])
    metrics=holdout_metrics(labels,probabilities,classes,positive_label)
    metrics["fold_count"]=len(folds)
    metrics["coverage_policy"]="Each eligible original train/development record held out once"
    metrics["interpretation"]="Pooled out-of-domain development predictions, not final-test scores"
    return metrics


### scripts/evaluate_generalization.py


In [ ]:
%%chainshield_file scripts/evaluate_generalization.py
"""Run frozen development-only source/event holdouts; resumable by verified fold."""
from __future__ import annotations
import argparse
import csv
from datetime import datetime,timezone
import json
import os
from pathlib import Path
import sys
import tempfile
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
os.environ["TEMP"]=str(ROOT/".tmp")
os.environ["TMP"]=str(ROOT/".tmp")
tempfile.tempdir=str(ROOT/".tmp")
from threadpoolctl import threadpool_limits
from chainshield.nlp.generalization import (
    load_pool,construct_fold,manifest_bytes,manifest_digest,holdout_metrics,aggregate_predictions,
)
from chainshield.nlp.baseline import build_text_baseline
from chainshield.ml.train import train_validate,save_artifact,sha256


def jobs_for(protocol,config):
    pools,jobs={},[]
    specs={(r["dataset"],r["task"]):r for r in config["nlp"]["tasks"]}
    for kind in ("source_transfer","event_holdout"):
        section=protocol[kind]
        dataset=section["dataset"]
        directory=ROOT/"data/processed/research-v2/paired_v2/control"/dataset
        label_map=json.loads((directory/"label_map.json").read_text(encoding="utf-8"))
        for task in section["tasks"]:
            pool=load_pool(directory,dataset,task,section.get("view"))
            classes=sorted(label_map[task] if dataset=="crisismmd" else set(label_map[task].values()))
            positive=specs[(dataset,task)].get("positive_label")
            pools[(dataset,task)]={"rows":pool,"classes":classes,"positive_label":positive}
            domains=sorted({r["domain"] for r in pool})
            if kind=="source_transfer":
                plans=[(r["training_domains"],r["heldout_domain"],
                    "train_"+"_".join(r["training_domains"])+"__holdout_"+r["heldout_domain"])
                    for r in section["directions"]]
                expected={v for r in section["directions"]
                          for v in r["training_domains"]+[r["heldout_domain"]]}
            else:
                expected=set(section["events"])
                plans=[([v for v in domains if v!=held],held,"heldout_"+held)
                       for held in section["events"]]
            if set(domains)!=expected:
                raise ValueError("Source/event list differs from declared protocol")
            for training,held,name in plans:
                fold=construct_fold(pool,training,held,classes)
                jobs.append({"dataset":dataset,"task":task,"kind":kind,"name":name,
                             "classes":classes,"positive_label":positive,"fold":fold})
    if len(jobs)!=protocol["expected_fold_count"]:
        raise ValueError("Declared fold-count mismatch")
    return pools,jobs


def read_completed(directory,expected,heldout):
    marker=json.loads((directory/"completion.json").read_text(encoding="utf-8"))
    required={"model.joblib","metadata.json","metrics.json","dev_predictions.csv","split_manifest.csv"}
    if set(marker["file_sha256"])!=required:
        raise ValueError("Incomplete fold completion record")
    for filename,digest in marker["file_sha256"].items():
        if sha256(directory/filename)!=digest:
            raise ValueError("Completed fold integrity failure: "+filename)
    metadata=json.loads((directory/"metadata.json").read_text(encoding="utf-8"))
    for key,value in expected.items():
        if metadata.get(key)!=value:
            raise ValueError("Completed fold no longer matches the protocol: "+key)
    metrics=json.loads((directory/"metrics.json").read_text(encoding="utf-8"))
    if metrics.get("test_evaluated") is not False or not metrics.get("saved_model_reload_verified"):
        raise ValueError("Completed fold lacks required evaluation evidence")
    with (directory/"dev_predictions.csv").open(encoding="utf-8",newline="") as handle:
        predictions=list(csv.DictReader(handle))
    expected_labels={r["record_id"]:r["label"] for r in heldout}
    if len(predictions)!=len(expected_labels) or {r["record_id"] for r in predictions}!=set(expected_labels):
        raise ValueError("Completed fold prediction coverage differs")
    if any(r["partition"]!="dev" or r["actual_label"]!=expected_labels[r["record_id"]]
           for r in predictions):
        raise ValueError("Invalid completed holdout predictions")
    return (
        {"artifact":directory.as_posix(),"metadata":metadata,"metrics":metrics,
         "completion_sha256":sha256(directory/"completion.json")},
        {"ids":[r["record_id"] for r in predictions],
         "labels":[r["actual_label"] for r in predictions],
         "classes":metadata["classes"],
         "probabilities":[[float(r["probability:"+label]) for label in metadata["classes"]]
                          for r in predictions]},
    )


def markdown(results,aggregate):
    lines=["# Source and event generalization validation","",
        "Fixed-setting TF-IDF/logistic experiments on eligible original train/development records only.",
        "Final test partitions were not fitted, scored, tuned, or exported.","",
        "## Protocol","",
        "- Six reciprocal source-transfer folds: original to India and India to original for three separate tasks.",
        "- Fourteen event folds: leave one of seven CrisisMMD events out, separately for two tasks.",
        "- Shared CrisisMMD view; all messages from the held-out event excluded from that fold's training.",
        "- Training records sharing exact normalized text or a connected group with held-out records are purged.",
        "- Previously fixed TF-IDF/logistic settings; no tuning on these holdout scores.",
        "- Complete original train/development pool held out exactly once per task across folds.",
        "- Mechanical templates and semantic near-duplicates are not automatically removed.",
        "- Protocol snapshot is saved before training; completed folds have integrity markers for resumption.","",
        "## Fold results","",
        "| Dataset / task | Held-out domain | Training | Held out | Purged | Global macro-F1 | Supported-class macro-F1 | Accuracy |",
        "|---|---|---:|---:|---:|---:|---:|---:|"]
    for result in results:
        meta,metrics=result["metadata"],result["metrics"]
        lines.append(f"| {meta['dataset']} / {meta['task']} | {meta['heldout_domain']} | "
            f"{metrics['training_records']} | {metrics['records']} | {meta['purged_training_rows']} | "
            f"{metrics['macro_f1']:.4f} | {metrics['macro_f1_supported_classes']:.4f} | {metrics['accuracy']:.4f} |")
    lines+=["","## Pooled out-of-domain development predictions","",
        "| Dataset / task | Folds | Held-out records | Macro-F1 | Accuracy |",
        "|---|---:|---:|---:|---:|"]
    for item in aggregate:
        metrics=item["metrics"]
        lines.append(f"| {item['dataset']} / {item['task']} | {metrics['fold_count']} | "
            f"{metrics['records']} | {metrics['macro_f1']:.4f} | {metrics['accuracy']:.4f} |")
    lines+=["","## Interpretation and limits","",
        "These are genuinely source/event-exclusive fitted models, rather than a breakdown of mixed-source validation predictions.",
        "Each fold excludes its own held-out domain; other folds may train on that domain as part of cross-validation.",
        "The original development data previously informed baseline comparisons. These stress experiments are not an untouched final test.",
        "Mixed-split validation and domain holdouts have different training/evaluation cohorts; score differences are not paired estimates of domain shift.",
        "Original-source is a provenance category, not proof of representative coverage of all Nepal or regional posts.",
        "Global macro-F1 includes declared classes absent from a particular held-out event; supported-class macro-F1 and per-class supports clarify that distinction.",
        "Small rare-class supports, rounded source IDs, templates, paraphrases and related events constrain generalization claims.",
        "High or low scores do not automatically change project scope, supported labels or original partitions.",
        "Apply this frozen protocol to neural comparisons. Final testing follows model/threshold/label decisions and experiment freeze.",
        "Files named dev_predictions.csv contain held-out-domain development predictions; their dev marker does not mean the original mixed-source dev partition.",
        "Every saved model passed probability-equivalence verification after reloading. Existing baseline artifacts were preserved.","",
        "## Training warnings",""]
    warnings_found=False
    for result in results:
        for warning in result["metrics"]["training_warnings"]:
            warnings_found=True
            lines.append("- "+result["metadata"]["model_version"]+": "+warning)
    if not warnings_found:
        lines.append("No training warnings recorded.")
    return "\n".join(lines)+"\n"


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--resume",action="store_true",help="Verify and reuse completed folds from an interrupted run")
    args=parser.parse_args()
    if Path(sys.prefix).resolve()!=(ROOT/".venv").resolve() or not (ROOT/".tmp").is_dir():
        raise ValueError("Use the managed Colab environment and temporary directory")
    protocol_path=ROOT/"configs/generalization.json"
    config_path=ROOT/"configs/training.json"
    protocol=json.loads(protocol_path.read_text(encoding="utf-8"))
    config=json.loads(config_path.read_text(encoding="utf-8"))
    if sha256(config_path)!=protocol["training_config_sha256"] or config["seed"]!=protocol["seed"]:
        raise ValueError("Frozen training configuration changed")
    if set(protocol["eligible_original_partitions"])!={"train","dev"}:
        raise ValueError("Final-test inclusion forbidden")
    destination=ROOT/"artifacts/generalization"
    report_path=ROOT/"docs/generalization-validation.md"
    if report_path.exists() or (destination/"summary.json").exists():
        raise FileExistsError("Generalization already completed; do not overwrite evidence")
    if destination.exists() and not args.resume:
        raise FileExistsError("Use --resume to verify an interrupted run")
    if args.resume and not (destination/"protocol.json").is_file():
        raise ValueError("No resumable protocol snapshot")
    pools,jobs=jobs_for(protocol,config)
    print(f"Declared {len(jobs)} folds; class-support and isolation preflight passed.",flush=True)
    implementation={name:sha256(ROOT/name) for name in (
        "src/chainshield/nlp/generalization.py","scripts/evaluate_generalization.py",
        "src/chainshield/nlp/baseline.py","src/chainshield/ml/train.py")}
    protected={}
    for dataset in ("india_nepal","crisismmd"):
        directory=ROOT/"data/processed/research-v2/paired_v2/control"/dataset
        quality=json.loads((directory/"quality.json").read_text(encoding="utf-8"))
        protected[(directory/"quality.json").relative_to(ROOT).as_posix()]=sha256(directory/"quality.json")
        for filename,digest in quality["output_sha256"].items():
            protected[(directory/filename).relative_to(ROOT).as_posix()]=digest
    baseline_path=ROOT/"artifacts/baselines/baseline_summary.json"
    protected[baseline_path.relative_to(ROOT).as_posix()]=sha256(baseline_path)
    baseline=json.loads(baseline_path.read_text(encoding="utf-8"))
    for result in baseline["results"]:
        path=Path(result["artifact"])/"model.joblib"
        path.resolve().relative_to(ROOT)
        protected[path.relative_to(ROOT).as_posix()]=result["metadata"]["artifact_sha256"]
    for name,digest in protected.items():
        if sha256(ROOT/name)!=digest:
            raise ValueError("Protected existing evidence changed: "+name)
    binding={"protocol_sha256":sha256(protocol_path),"implementation_sha256":implementation,
             "protected_files_sha256":protected,"training_config":config}
    if args.resume:
        snapshot=json.loads((destination/"protocol.json").read_text(encoding="utf-8"))
        if snapshot["binding"]!=binding:
            raise ValueError("Resume binding differs from the original run")
    else:
        destination.mkdir(parents=True)
        with (destination/"protocol.json").open("x",encoding="utf-8") as handle:
            json.dump({"declared_at_utc":datetime.now(timezone.utc).isoformat(),
                       "protocol":protocol,"binding":binding},handle,indent=2)
            handle.write("\n")
    results=[]
    predictions={key:[] for key in pools}
    with threadpool_limits(limits=protocol["threads"]):
        for number,job in enumerate(jobs,1):
            dataset,task,name=job["dataset"],job["task"],job["name"]
            fold=job["fold"]
            train,held=fold["training"],fold["heldout"]
            output=destination/dataset/task/name
            expected={"dataset":dataset,"task":task,"classes":job["classes"],
                "model_name":"tfidf_logistic_regression",
                "model_version":protocol["protocol_id"]+"/"+dataset+"/"+task+"/"+name,
                "fold_name":name,"kind":job["kind"],"training_domains":fold["training_domains"],
                "heldout_domain":fold["heldout_domain"],"purged_training_rows":fold["purged_training_rows"],
                "invariants":fold["invariants"],"split_plan_sha256":manifest_digest(fold["manifest"]),
                "protocol_sha256":sha256(protocol_path),"training_config_sha256":sha256(config_path),
                "source_quality_sha256":sha256(ROOT/"data/processed/research-v2/paired_v2/control"/dataset/"quality.json"),
                "development_only":True,"test_evaluated":False}
            print(f"[{number}/{len(jobs)}] {dataset}/{task}/{name}: train={len(train)}, heldout={len(held)}",flush=True)
            if output.exists():
                if not args.resume or not (output/"completion.json").is_file():
                    raise ValueError("Partial fold found; inspect rather than overwrite: "+str(output))
                result,fold_predictions=read_completed(output,expected,held)
                print("  Reused integrity-verified completed fold",flush=True)
            else:
                model=build_text_baseline(config)
                X_train=[r["text"] for r in train]
                y_train=[r["label"] for r in train]
                X_held=[r["text"] for r in held]
                labels=[r["label"] for r in held]
                ids=[r["record_id"] for r in held]
                metrics,probabilities=train_validate(model,X_train,y_train,X_held,labels,
                    job["positive_label"],config["latency_samples"])
                metrics.update(holdout_metrics(labels,probabilities,model.classes_,job["positive_label"]))
                if model.classes_.tolist()!=job["classes"]:
                    raise ValueError("Trained classes differ from declared ordering")
                metadata=dict(expected,training_label_counts=fold["training_label_counts"],
                              heldout_label_counts=fold["heldout_label_counts"],seed=protocol["seed"])
                result=save_artifact(output,model,metadata,metrics,ids,labels,probabilities,X_held)
                with (output/"split_manifest.csv").open("xb") as handle:
                    handle.write(manifest_bytes(fold["manifest"]))
                with (output/"completion.json").open("x",encoding="utf-8") as handle:
                    json.dump({"completed_at_utc":datetime.now(timezone.utc).isoformat(),
                        "file_sha256":{filename:sha256(output/filename) for filename in (
                        "model.joblib","metadata.json","metrics.json","dev_predictions.csv","split_manifest.csv")}},
                        handle,indent=2)
                    handle.write("\n")
                result["completion_sha256"]=sha256(output/"completion.json")
                fold_predictions={"ids":ids,"labels":labels,"classes":model.classes_.tolist(),
                                  "probabilities":probabilities}
            results.append(result)
            predictions[(dataset,task)].append(fold_predictions)
            print(f"  Held-out macro-F1={result['metrics']['macro_f1']:.4f}",flush=True)
    aggregate=[]
    for (dataset,task),pool in pools.items():
        metrics=aggregate_predictions(predictions[(dataset,task)],
            [r["record_id"] for r in pool["rows"]],pool["classes"],pool["positive_label"])
        aggregate.append({"dataset":dataset,"task":task,"metrics":metrics})
    for name,digest in protected.items():
        if sha256(ROOT/name)!=digest:
            raise ValueError("Existing evidence changed during the experiment: "+name)
    summary={"schema_version":1,"protocol_id":protocol["protocol_id"],
        "created_at_utc":datetime.now(timezone.utc).isoformat(),
        "protocol_sha256":sha256(protocol_path),"fold_count":len(results),
        "test_evaluated":False,"original_datasets_changed":False,"original_baselines_changed":False,
        "results":results,"aggregate":aggregate}
    with (destination/"summary.json").open("x",encoding="utf-8") as handle:
        json.dump(summary,handle,indent=2,allow_nan=False)
        handle.write("\n")
    with report_path.open("x",encoding="utf-8") as handle:
        handle.write(markdown(results,aggregate))
    print(f"Complete: {len(results)} domain-exclusive models; reload, coverage and evidence-preservation checks passed.",flush=True)


if __name__=="__main__":
    main()


### configs/neural.json


In [ ]:
%%chainshield_file configs/neural.json
{
  "schema_version": 1,
  "experiment_id": "research-v2-neural",
  "seed": 42,
  "threads": 2,
  "partitions": [
    "train",
    "dev"
  ],
  "generalization_protocol": "configs/generalization.json",
  "pretrained": {
    "repository": "distilbert/distilbert-base-uncased",
    "revision": "12040accade4e8a0f71eabdb258fecc2e7e948be",
    "directory": "artifacts/neural/pretrained/distilbert"
  },
  "maximum_length": 128,
  "checkpoint_optimizer_steps": 200,
  "gradient_clip": 1,
  "warmup_fraction": 0.1,
  "bilstm": {
    "maximum_vocabulary": 20000,
    "minimum_frequency": 2,
    "embedding_size": 128,
    "hidden_size": 128,
    "dropout": 0.3,
    "batch_size": 32,
    "gradient_accumulation": 1,
    "epochs": 10,
    "learning_rate": 0.001,
    "weight_decay": 0.0001
  },
  "distilbert": {
    "batch_size": 4,
    "gradient_accumulation": 4,
    "epochs": 3,
    "learning_rate": 2e-05,
    "weight_decay": 0.01
  },
  "class_weight_policy": "Inverse-frequency weights computed from the current training fold only",
  "main_selection_policy": "Best development macro-F1 among fixed configured epochs",
  "domain_selection_policy": "Last configured epoch; held-out domain evaluated only after training",
  "tasks": [
    {
      "id": "screening",
      "model": "bilstm",
      "dataset": "crisismmd",
      "task": "informative",
      "view": "shared",
      "positive_label": "informative"
    },
    {
      "id": "disaster_type",
      "model": "distilbert",
      "dataset": "india_nepal",
      "task": "disaster_type",
      "positive_label": "earthquake"
    },
    {
      "id": "humanitarian",
      "model": "distilbert",
      "dataset": "crisismmd",
      "task": "humanitarian",
      "view": "shared"
    }
  ],
  "final_test_policy": "No fitting, scoring, tuning or prediction export for final-test records",
  "expected_main_models": 3,
  "expected_generalization_models": 16,
  "generalization_protocol_sha256": "cb3512f6aad3a220dde4ca6fabce67ba7de46f53af85010f0e535cf1c0bd470f",
  "implementation_components_sha256": {
    "src/chainshield/nlp/bilstm.py": "d06c4e8e3fcc0e678e8500b6fa7d2942200984090bcf4b938fc076df76931266",
    "src/chainshield/nlp/transformer.py": "57ac1c7d3773038cdd6a3001902f34698e2f87a3da35440e81de40c6b2876994",
    "src/chainshield/nlp/neural_train.py": "0f9226a1d7318c86143fdcd12d488274900f4cf69f1d2f68b00e54a152428006",
    "src/chainshield/nlp/generalization.py": "7301ca723ab09a4f7c3528f48d1f0df188ccaef5ec197499638ceee72d9ca08e",
    "src/chainshield/ml/train.py": "99f776c81d140b734ddd1576732dd89adfa131b91402d956ee212260835e3c4e"
  }
}


### src/chainshield/nlp/neural_train.py


In [ ]:
%%chainshield_file src/chainshield/nlp/neural_train.py
"""Resumable local neural training, safe final weights, and domain-aware inference."""
from __future__ import annotations
from collections import Counter
import csv
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import random
from time import perf_counter
import numpy as np
import torch
from safetensors.torch import save_file,load_file
from chainshield.ml.train import sha256
from chainshield.nlp import bilstm
from chainshield.nlp.generalization import holdout_metrics


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)


def training_class_weights(labels,classes):
    counts=Counter(labels)
    if set(counts)!=set(classes) or not classes:
        raise ValueError("Every declared class must occur in the current training fold")
    return torch.tensor([len(labels)/(len(classes)*counts[label]) for label in classes],
                        dtype=torch.float32)


def batch_inputs(encoded,indexes,device):
    selected={key:value[indexes] for key,value in encoded.items()}
    width=max(1,int(selected["attention_mask"].sum(dim=1).max()))
    return {key:value[:,:width].to(device) for key,value in selected.items()}


def logits(model,inputs):
    output=model(**inputs)
    return output.logits if hasattr(output,"logits") else output


def probabilities(model,encoded,device,batch_size):
    model.eval()
    result=[]
    with torch.inference_mode():
        for start in range(0,len(encoded["input_ids"]),batch_size):
            indexes=torch.arange(start,min(start+batch_size,len(encoded["input_ids"])))
            inputs=batch_inputs(encoded,indexes,device)
            scores=torch.softmax(logits(model,inputs).float(),dim=-1)
            result.append(scores.cpu().numpy())
    return np.vstack(result)


def atomic_json(path,value):
    temporary=Path(str(path)+".pending")
    with temporary.open("w",encoding="utf-8") as handle:
        json.dump(value,handle,indent=2,allow_nan=False)
        handle.write("\n")
    os.replace(temporary,path)


def checkpoint_save(directory,state,binding):
    directory=Path(directory)
    directory.mkdir(parents=True,exist_ok=True)
    slot="a" if state["checkpoint_generation"]%2==0 else "b"
    target=directory/("checkpoint-"+slot+".pt")
    pending=directory/("checkpoint-"+slot+".pending")
    torch.save(state,pending)
    os.replace(pending,target)
    atomic_json(directory/("checkpoint-"+slot+".json"),{
        "generation":state["checkpoint_generation"],"binding":binding,
        "sha256":sha256(target),"global_step":state["global_step"]})


def checkpoint_load(directory,binding):
    candidates=[]
    for slot in ("a","b"):
        marker=Path(directory)/("checkpoint-"+slot+".json")
        path=Path(directory)/("checkpoint-"+slot+".pt")
        if not marker.exists() or not path.exists():
            continue
        record=json.loads(marker.read_text(encoding="utf-8"))
        if record["binding"]!=binding:
            raise ValueError("Checkpoint configuration/input binding differs")
        if sha256(path)==record["sha256"]:
            candidates.append((record["generation"],path))
    if not candidates:
        if any(Path(directory).glob("checkpoint-*.pt")):
            raise ValueError("No integrity-valid checkpoint slot; preserve files for inspection")
        return None
    _,path=max(candidates,key=lambda value:value[0])
    return torch.load(path,map_location="cpu",weights_only=True)


def rng_state():
    return {"torch":torch.get_rng_state(),
            "cuda":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "python":random.getstate()}


def restore_rng(state):
    torch.set_rng_state(state["torch"])
    random.setstate(state["python"])
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])


def train_network(model,encoded,labels,classes,config,spec,directory,binding,device,
                  *,validation=None,positive_label=None,step_budget=None):
    """Main runs select best development epoch; domain runs evaluate only at the end."""
    seed_everything(config["seed"])
    model.to(device)
    class_index={label:i for i,label in enumerate(classes)}
    targets=torch.tensor([class_index[label] for label in labels],dtype=torch.long)
    weight=training_class_weights(labels,classes).to(device)
    criterion=torch.nn.CrossEntropyLoss(weight=weight)
    optimizer=torch.optim.AdamW(model.parameters(),lr=spec["learning_rate"],
                                weight_decay=spec["weight_decay"])
    batches=math.ceil(len(labels)/spec["batch_size"])
    accumulation=spec["gradient_accumulation"]
    steps_per_epoch=math.ceil(batches/accumulation)
    total_steps=steps_per_epoch*spec["epochs"]
    warmup=max(1,int(total_steps*config["warmup_fraction"]))
    def learning_rate(step):
        return (step+1)/warmup if step<warmup else max(0.0,(total_steps-step)/max(1,total_steps-warmup))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,learning_rate)
    scaler=torch.amp.GradScaler("cuda",enabled=device.type=="cuda")
    state=checkpoint_load(directory,binding)
    if state is None:
        state={"epoch":0,"next_batch":0,"global_step":0,"history":[],
            "best_score":None,"best_epoch":None,"best_state":None,"loss_sum":0.0,
            "batch_count":0,"elapsed_seconds":0.0,"checkpoint_generation":0,
            "skipped_amp_steps":0}
    else:
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        restore_rng(state["rng"])
        print(f"  Resumed epoch {state['epoch']+1}, next batch {state['next_batch']}, step {state['global_step']}",flush=True)
    elapsed_before=state["elapsed_seconds"]
    session_start=perf_counter()
    def persist():
        state["model"]=model.state_dict()
        state["optimizer"]=optimizer.state_dict()
        state["scheduler"]=scheduler.state_dict()
        state["scaler"]=scaler.state_dict()
        state["rng"]=rng_state()
        state["elapsed_seconds"]=elapsed_before+perf_counter()-session_start
        state["checkpoint_generation"]+=1
        checkpoint_save(directory,state,binding)
    if state["global_step"]==0 and state["epoch"]==0:
        persist()
    optimizer.zero_grad(set_to_none=True)
    starting_epoch=state["epoch"]
    for epoch in range(starting_epoch,spec["epochs"]):
        model.train()
        generator=torch.Generator().manual_seed(config["seed"]+epoch)
        permutation=torch.randperm(len(labels),generator=generator)
        start_batch=state["next_batch"] if epoch==starting_epoch else 0
        epoch_started=perf_counter()
        for batch in range(start_batch,batches):
            indexes=permutation[batch*spec["batch_size"]:(batch+1)*spec["batch_size"]]
            inputs=batch_inputs(encoded,indexes,device)
            group_start=(batch//accumulation)*accumulation
            group_size=min(accumulation,batches-group_start)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type=="cuda"):
                loss=criterion(logits(model,inputs),targets[indexes].to(device))
                scaled_loss=loss/group_size
            scaler.scale(scaled_loss).backward()
            state["loss_sum"]+=float(loss.detach().cpu())
            state["batch_count"]+=1
            state["epoch"]=epoch
            state["next_batch"]=batch+1
            if (batch+1)%accumulation==0 or batch+1==batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),config["gradient_clip"])
                old_scale=scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale()>=old_scale:
                    scheduler.step()
                else:
                    state["skipped_amp_steps"]+=1
                optimizer.zero_grad(set_to_none=True)
                state["global_step"]+=1
                if state["global_step"]%25==0:
                    print(f"  Epoch {epoch+1}/{spec['epochs']}, batch {batch+1}/{batches}, "
                          f"step {state['global_step']}, mean loss {state['loss_sum']/state['batch_count']:.4f}",flush=True)
                if state["global_step"]%config["checkpoint_optimizer_steps"]==0:
                    persist()
                if step_budget is not None and state["global_step"]>=step_budget:
                    persist()
                    return {"completed":False,"global_step":state["global_step"]}
        record={"epoch":epoch+1,"mean_batch_loss":state["loss_sum"]/max(1,state["batch_count"]),
                "epoch_session_seconds":perf_counter()-epoch_started}
        if validation is not None:
            dev_probs=probabilities(model,validation["encoded"],device,spec["batch_size"])
            metrics=holdout_metrics(validation["labels"],dev_probs,classes,positive_label)
            record["development_macro_f1"]=metrics["macro_f1"]
            indexes=[classes.index(y) for y in validation["labels"]]
            record["development_log_loss"]=float(-np.log(np.clip(dev_probs[np.arange(len(indexes)),indexes],1e-15,1)).mean())
            record["loss_definitions"]={"train":"Arithmetic mean of weighted cross-entropy minibatch losses","development":"Unweighted mean negative log probability of true labels; not the same objective as weighted training loss"}
            if state["best_score"] is None or metrics["macro_f1"]>state["best_score"]:
                state["best_score"]=metrics["macro_f1"]
                state["best_epoch"]=epoch+1
                state["best_state"]={name:value.detach().cpu().clone()
                                     for name,value in model.state_dict().items()}
            print(f"  Epoch {epoch+1} development macro-F1={metrics['macro_f1']:.4f}",flush=True)
        state["history"].append(record)
        state["epoch"]=epoch+1
        state["next_batch"]=0
        state["loss_sum"]=0.0
        state["batch_count"]=0
        persist()
    if validation is not None:
        if state["best_state"] is None:
            raise ValueError("No selected development checkpoint")
        model.load_state_dict(state["best_state"])
    model.eval()
    return {"completed":True,"history":state["history"],
        "trainable_parameters":sum(p.numel() for p in model.parameters() if p.requires_grad),
        "selected_epoch":state["best_epoch"] if validation is not None else spec["epochs"],
        "selection":"best_development_macro_f1" if validation is not None else "fixed_last_epoch",
        "training_wall_seconds":elapsed_before+perf_counter()-session_start,
        "global_step":state["global_step"],"skipped_amp_steps":state["skipped_amp_steps"]}


def make_components(kind,texts,classes,config,pretrained,vocabulary=None,from_pretrained=True):
    if kind=="bilstm":
        vocabulary=vocabulary or bilstm.build_vocabulary(texts,
            config["bilstm"]["maximum_vocabulary"],config["bilstm"]["minimum_frequency"])
        model=bilstm.make_model(vocabulary,classes,config["bilstm"])
        return model,vocabulary,lambda values:bilstm.encode_texts(values,vocabulary,config["maximum_length"])
    from chainshield.nlp import transformer
    tokenizer=transformer.make_tokenizer(pretrained)
    model=transformer.make_model(pretrained,classes,from_pretrained=from_pretrained)
    return model,None,lambda values:transformer.encode_texts(values,tokenizer,config["maximum_length"])


def save_final_model(model,directory,metadata,vocabulary=None):
    directory=Path(directory)
    path=directory/"model.safetensors"
    if path.exists():
        raise FileExistsError("Final neural model already exists")
    weights={name:value.detach().cpu().contiguous() for name,value in model.state_dict().items()}
    save_file(weights,str(path))
    metadata["weights_sha256"]=sha256(path)
    if vocabulary is not None:
        with (directory/"vocabulary.json").open("x",encoding="utf-8") as handle:
            json.dump(vocabulary,handle,ensure_ascii=False,sort_keys=True)
            handle.write("\n")
        metadata["vocabulary_sha256"]=sha256(directory/"vocabulary.json")
    metadata["libraries"]={name:importlib.metadata.version(name)
                          for name in ("torch","transformers","numpy","safetensors")}
    with (directory/"metadata.json").open("x",encoding="utf-8") as handle:
        json.dump(metadata,handle,indent=2)
        handle.write("\n")


def load_final_model(directory,device="cpu"):
    directory=Path(directory)
    metadata=json.loads((directory/"metadata.json").read_text(encoding="utf-8"))
    if sha256(directory/"model.safetensors")!=metadata["weights_sha256"]:
        raise ValueError("Neural weights failed integrity verification")
    vocabulary=None
    if metadata["model"]=="bilstm":
        if sha256(directory/"vocabulary.json")!=metadata["vocabulary_sha256"]:
            raise ValueError("Vocabulary failed integrity verification")
        vocabulary=json.loads((directory/"vocabulary.json").read_text(encoding="utf-8"))
    if metadata["model"]=="distilbert":
        base=Path(metadata["pretrained_directory"])
        if sha256(base/"manifest.json")!=metadata["pretrained_manifest_sha256"]:
            raise ValueError("Pinned pretrained metadata changed")
        assets=json.loads((base/"manifest.json").read_text(encoding="utf-8"))
        for entry in assets["files"]:
            if entry["file"] in {"config.json","tokenizer.json","tokenizer_config.json","vocab.txt"}:
                if sha256(base/entry["file"])!=entry["sha256"]:
                    raise ValueError("Pinned tokenizer/config asset changed")
    model,_,encode=make_components(metadata["model"],[],metadata["classes"],metadata["config"],
        metadata["pretrained_directory"],vocabulary=vocabulary,from_pretrained=False)
    model.load_state_dict(load_file(str(directory/"model.safetensors")))
    model.to(device)
    model.eval()
    return model,metadata,encode


def predict_text(text,directory,*,domain_eligible=False,minimum_confidence=0.6):
    if not isinstance(text,str) or not text.strip() or not isinstance(domain_eligible,bool):
        raise ValueError("Nonempty text and explicit domain eligibility are required")
    if not 0<=minimum_confidence<=1:
        raise ValueError("Invalid confidence threshold")
    model,metadata,encode=load_final_model(directory)
    from chainshield.research.preparation import normalize_urls_mentions
    if metadata.get("preprocessing_variant")=="url_mentions":text=normalize_urls_mentions(text)
    probs=probabilities(model,encode([text]),torch.device("cpu"),1)[0]
    index=int(np.argmax(probs))
    score=float(probs[index])
    accepted=domain_eligible and score>=minimum_confidence
    return {"task":metadata["task"],"model_version":metadata["model_version"],
        "label":metadata["classes"][index] if accepted else None,
        "candidate_label":metadata["classes"][index],"score":score,
        "status":"predicted" if accepted else "review_required",
        "reason":"" if accepted else "Domain eligibility or confidence requires review",
        "verification_status":"unverified_report","shipment_exposure":"unknown",
        "score_interpretation":"Uncalibrated classifier probability, not incident verification"}


### scripts/train_neural.py


In [ ]:
%%chainshield_file scripts/train_neural.py
"""Train primary and frozen-protocol neural models without final-test evaluation."""
from __future__ import annotations
import argparse
import csv
from datetime import datetime,timezone
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import sys
import tempfile
from time import perf_counter
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
os.environ["TEMP"]=str(ROOT/".tmp")
os.environ["TMP"]=str(ROOT/".tmp")
os.environ["HF_HOME"]=str(ROOT/"artifacts/neural/cache")
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
tempfile.tempdir=str(ROOT/".tmp")
import numpy as np
import torch
from chainshield.ml.train import sha256
from chainshield.nlp.baseline import load_text_partitions
from chainshield.nlp.generalization import load_pool,construct_fold,manifest_bytes,aggregate_predictions,holdout_metrics
from chainshield.nlp.neural_train import (
    seed_everything,make_components,train_network,probabilities,save_final_model,load_final_model,
)



def canonical_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":")).encode()).hexdigest()


def plans(config,protocol):
    jobs=[]
    pools={}
    for task in config["tasks"]:
        dataset,name=task["dataset"],task["task"]
        directory=ROOT/"data/processed/research-v2/paired_v2/control"/dataset
        label_map=json.loads((directory/"label_map.json").read_text(encoding="utf-8"))
        classes=sorted(label_map[name] if dataset=="crisismmd" else set(label_map[name].values()))
        selected_path=ROOT/"artifacts/research-v2/search"/dataset/name/"selection.json"
        selection=json.loads(selected_path.read_text())
        variant=selection["candidate"].get("variant","control")
        main_directory=ROOT/"data/processed/research-v2/paired_v2"/variant/dataset
        parts=load_text_partitions(main_directory,dataset,name,task.get("view"))
        main_manifest=[]
        for split,role in (("train","training"),("dev","heldout_development")):
            part=parts[split]
            for identifier,label,group in zip(part["ids"],part["labels"],part["groups"]):
                main_manifest.append({"record_id":identifier,"label":label,"group_id":group,
                                      "original_partition":split,"role":role})
        jobs.append({"id":task["id"],"model":task["model"],"dataset":dataset,"task":name,
            "classes":classes,"positive_label":task.get("positive_label"),"kind":"main",
            "name":"main","preprocessing_variant":variant,"train":parts["train"],"evaluation":parts["dev"],
            "manifest":main_manifest})
        pool=load_pool(directory,dataset,name,task.get("view"))
        pools[task["id"]]={"rows":pool,"classes":classes,"positive_label":task.get("positive_label"),
                          "dataset":dataset,"task":name,"model":task["model"]}
        if dataset=="india_nepal":
            section=protocol["source_transfer"]
            specifications=[(entry["training_domains"],entry["heldout_domain"],
                "train_"+"_".join(entry["training_domains"])+"__holdout_"+entry["heldout_domain"])
                for entry in section["directions"]]
        else:
            events=protocol["event_holdout"]["events"]
            if set(r["domain"] for r in pool)!=set(events):
                raise ValueError("Events differ from frozen baseline protocol")
            specifications=[([event for event in events if event!=held],held,"heldout_"+held) for held in events]
        for training,held,name_fold in specifications:
            fold=construct_fold(pool,training,held,classes)
            def part(rows):
                return {"ids":[r["record_id"] for r in rows],"texts":[r["text"] for r in rows],
                        "labels":[r["label"] for r in rows],"groups":[r["group_id"] for r in rows]}
            jobs.append({"id":task["id"],"model":task["model"],"dataset":dataset,"task":name,
                "classes":classes,"positive_label":task.get("positive_label"),"kind":"generalization",
                "name":name_fold,"train":part(fold["training"]),"evaluation":part(fold["heldout"]),
                "manifest":fold["manifest"],"heldout_domain":held,"training_domains":training,
                "purged_training_rows":fold["purged_training_rows"]})
    main=[job for job in jobs if job["kind"]=="main"]
    general=[job for job in jobs if job["kind"]=="generalization"]
    if len(main)!=config["expected_main_models"] or len(general)!=config["expected_generalization_models"]:
        raise ValueError("Declared neural job count differs")
    return main+general,pools


def read_complete(directory,binding,job):
    marker=json.loads((directory/"completion.json").read_text(encoding="utf-8"))
    if marker["job_binding_sha256"]!=canonical_hash(binding):
        raise ValueError("Completed neural job differs from frozen inputs")
    for filename,digest in marker["file_sha256"].items():
        if sha256(directory/filename)!=digest:
            raise ValueError("Neural artifact integrity mismatch: "+filename)
    meta=json.loads((directory/"metadata.json").read_text(encoding="utf-8"))
    metrics=json.loads((directory/"metrics.json").read_text(encoding="utf-8"))
    with (directory/"dev_predictions.csv").open(encoding="utf-8",newline="") as handle:
        rows=list(csv.DictReader(handle))
    expected=dict(zip(job["evaluation"]["ids"],job["evaluation"]["labels"]))
    if len(rows)!=len(expected) or {r["record_id"] for r in rows}!=set(expected):
        raise ValueError("Completed neural prediction coverage differs")
    if any(r["partition"]!="dev" or r["actual_label"]!=expected[r["record_id"]] for r in rows):
        raise ValueError("Completed predictions are not held-out development records")
    return {"artifact":directory.as_posix(),"metadata":meta,"metrics":metrics,
            "completion_sha256":sha256(directory/"completion.json")},{
        "ids":[r["record_id"] for r in rows],"labels":[r["actual_label"] for r in rows],
        "classes":meta["classes"],"probabilities":[[float(r["probability:"+label])
             for label in meta["classes"]] for r in rows]}


def report(results,aggregate,device):
    lines=["# Neural model experiments","",
        "Primary development comparisons and fixed-protocol source/event-held-out stress experiments.",
        "Final test partitions were not fitted, scored, tuned, or exported.",
        "Runtime device: "+str(device)+".","","## Primary models","",
        "| Task | Model | Train | Dev | Macro-F1 | Selected epoch |",
        "|---|---|---:|---:|---:|---:|"]
    for result in results:
        m,meta=result["metrics"],result["metadata"]
        if meta["kind"]=="main":
            lines.append(f"| {meta['dataset']} / {meta['task']} | {meta['model']} | "
                f"{m['training_records']} | {m['records']} | {m['macro_f1']:.4f} | {m['selected_epoch']} |")
    lines+=["","## Source/event-held-out neural results","",
        "| Task | Model | Held-out domain | Records | Macro-F1 |",
        "|---|---|---|---:|---:|"]
    for result in results:
        m,meta=result["metrics"],result["metadata"]
        if meta["kind"]=="generalization":
            lines.append(f"| {meta['task']} | {meta['model']} | {meta['heldout_domain']} | "
                         f"{m['records']} | {m['macro_f1']:.4f} |")
    lines+=["","## Pooled out-of-domain development predictions","",
        "| Task | Model | Folds | Records | Macro-F1 |",
        "|---|---|---:|---:|---:|"]
    for item in aggregate:
        m=item["metrics"]
        lines.append(f"| {item['task']} | {item['model']} | {m['fold_count']} | "
                     f"{m['records']} | {m['macro_f1']:.4f} |")
    lines+=["","## Controls and limitations","",
        "- BiLSTM vocabulary and class weights are fitted on the current training fold only.",
        "- DistilBERT uses a pinned official checkpoint with local-only loading and no remote code.",
        "- All DistilBERT parameters are fine-tuned; these are not random or frozen-feature substitutes.",
        "- Primary models select the best epoch on development macro-F1 among fixed configured epochs.",
        "- Domain-held-out models use the fixed last epoch and never select epochs using held-out scores.",
        "- Exact connected-group and normalized-text isolation follows the frozen baseline protocol.",
        "- Epoch/batch checkpoints preserve optimizer, scheduler, scaler and random-number state.",
        "- Final models use safetensors; local training checkpoints use restricted weights-only deserialization.",
        "- Every final model is reloaded and its evaluation probabilities compared with the original.",
        "- Text probabilities are uncalibrated; unknown domain eligibility defaults to manual review.",
        "- Templates, rare classes, source artifacts and differing cohort composition still constrain conclusions.",
        "- Neural methods need not beat TF-IDF. Actual results guide later model selection and application routing.",
        "- No final-test performance, verified incident truth, shipment exposure or operational benefit is established.",""]
    return "\n".join(lines)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--task",default="all",choices=("all","screening","disaster_type","humanitarian"))
    parser.add_argument("--phase",default="all",choices=("all","main","generalization"))
    args=parser.parse_args()
    if Path(sys.prefix).resolve()!=(ROOT/".venv").resolve():
        raise ValueError("Use the managed Colab environment")
    config_path=ROOT/"configs/neural.json"
    config=json.loads(config_path.read_text(encoding="utf-8"))
    protocol_path=ROOT/config["generalization_protocol"]
    if sha256(protocol_path)!=config["generalization_protocol_sha256"]:
        raise ValueError("Frozen generalization protocol changed")
    protocol=json.loads(protocol_path.read_text(encoding="utf-8"))
    if set(config["partitions"])!={"train","dev"}:
        raise ValueError("Final-test inclusion forbidden")
    torch.set_num_threads(config["threads"])
    device=torch.device("cpu")
    if device.type=="cuda":
        sample=torch.ones(4,device=device)
        if float((sample*sample).sum().cpu())!=4:
            raise ValueError("CUDA tensor check failed")
    jobs,pools=plans(config,protocol)
    selected=[job for job in jobs if (args.task=="all" or job["id"]==args.task) and
              (args.phase=="all" or job["kind"]==args.phase)]
    pretrained=ROOT/config["pretrained"]["directory"]
    manifest_hash=None
    if any(job["model"]=="distilbert" for job in selected):
        manifest_path=pretrained/"manifest.json"
        if not manifest_path.is_file():
            raise ValueError("Complete notebook 08 pretrained acquisition first")
        manifest=json.loads(manifest_path.read_text())
        if manifest["revision"]!=config["pretrained"]["revision"]:
            raise ValueError("Pretrained revision differs")
        for item in manifest["files"]:
            if sha256(pretrained/item["file"])!=item["sha256"]:
                raise ValueError("Pinned asset changed; no implicit network fallback")
        manifest_hash=sha256(manifest_path)
    elif (pretrained/"manifest.json").exists():
        manifest_hash=sha256(pretrained/"manifest.json")
    versions={name:importlib.metadata.version(name) for name in
              ("torch","transformers","numpy","pandas","scikit-learn","safetensors")}
    code=dict(config["implementation_components_sha256"])
    for name,expected in code.items():
        if not (ROOT/name).exists() and name=="src/chainshield/nlp/transformer.py" and not any(j["model"]=="distilbert" for j in selected):
            continue
        if sha256(ROOT/name)!=expected:
            raise ValueError("Neural component implementation differs: "+name)
    code["scripts/train_neural.py"]=sha256(Path(__file__))
    protected={}
    for filename in ("artifacts/baselines/baseline_summary.json","artifacts/generalization/summary.json"):
        protected[filename]=sha256(ROOT/filename)
    for dataset in ("india_nepal","crisismmd"):
      for variant in ("control","url_mentions"):
        directory=ROOT/"data/processed/research-v2/paired_v2"/variant/dataset
        quality=json.loads((directory/"quality.json").read_text(encoding="utf-8"))
        protected[(directory/"quality.json").relative_to(ROOT).as_posix()]=sha256(directory/"quality.json")
        for filename,digest in quality["output_sha256"].items():
            protected[(directory/filename).relative_to(ROOT).as_posix()]=digest
    for selected_path in (ROOT/"artifacts/research-v2/search").glob("*/*/selection.json"):
        protected[selected_path.relative_to(ROOT).as_posix()]=sha256(selected_path)
    binding={"config_sha256":sha256(config_path),"protocol_sha256":sha256(protocol_path),
             "pretrained_spec":config["pretrained"],"code_sha256":code,
             "libraries":versions,"device_type":device.type,"protected_files_sha256":protected}
    base=ROOT/"artifacts/neural"
    base.mkdir(parents=True,exist_ok=True)
    snapshot=base/"run-binding.json"
    if snapshot.exists():
        if json.loads(snapshot.read_text(encoding="utf-8"))!=binding:
            raise ValueError("Neural run binding changed; preserve existing training artifacts")
    else:
        with snapshot.open("x",encoding="utf-8") as handle:
            json.dump(binding,handle,indent=2)
            handle.write("\n")
    print(f"Neural device: {device}; selected {len(selected)} of {len(jobs)} declared jobs",flush=True)
    for number,job in enumerate(selected,1):
        directory=base/job["kind"]/job["id"]/job["name"]
        job_binding={"run_binding_sha256":canonical_hash(binding),
                     "split_manifest_sha256":hashlib.sha256(manifest_bytes(job["manifest"])).hexdigest(),
                     "id":job["id"],"model":job["model"],"kind":job["kind"],
                     "name":job["name"],"classes":job["classes"],
                     "pretrained_manifest_sha256":manifest_hash if job["model"]=="distilbert" else None}
        if (directory/"completion.json").exists():
            read_complete(directory,job_binding,job)
            print(f"[{number}/{len(selected)}] Reused verified {job['id']}/{job['name']}",flush=True)
            continue
        if any((directory/name).exists() for name in ("model.safetensors","metadata.json","metrics.json")):
            raise ValueError("Partial final artifact requires inspection: "+str(directory))
        directory.mkdir(parents=True,exist_ok=True)
        plan_path=directory/"job.json"
        if plan_path.exists():
            if json.loads(plan_path.read_text(encoding="utf-8"))!=job_binding:
                raise ValueError("Neural job inputs changed")
        else:
            with plan_path.open("x",encoding="utf-8") as handle:
                json.dump(job_binding,handle,indent=2)
        seed_everything(config["seed"])
        model,vocabulary,encode=make_components(job["model"],job["train"]["texts"],
            job["classes"],config,pretrained)
        train_encoded=encode(job["train"]["texts"])
        evaluation_encoded=encode(job["evaluation"]["texts"])
        spec=config[job["model"]]
        validation={"encoded":evaluation_encoded,"labels":job["evaluation"]["labels"]} if job["kind"]=="main" else None
        print(f"[{number}/{len(selected)}] Training {job['id']}/{job['name']}: "
              f"train={len(job['train']['ids'])}, evaluation={len(job['evaluation']['ids'])}",flush=True)
        trained=train_network(model,train_encoded,job["train"]["labels"],job["classes"],
            config,spec,directory,job_binding,device,validation=validation,
            positive_label=job["positive_label"])
        if not trained["completed"]:
            raise ValueError("Training did not complete")
        scores=probabilities(model,evaluation_encoded,device,spec["batch_size"])
        metrics=holdout_metrics(job["evaluation"]["labels"],scores,job["classes"],job["positive_label"])
        metrics.update({key:value for key,value in trained.items() if key not in ("history","completed")})
        metrics["evaluation_design"]="original mixed-source development partition" if job["kind"]=="main" else metrics["evaluation_design"]
        metrics["training_records"]=len(job["train"]["ids"])
        metrics["device_type"]=device.type
        timings=[]
        for index in range(min(20,len(job["evaluation"]["ids"]))):
            one={name:value[index:index+1] for name,value in evaluation_encoded.items()}
            started=perf_counter()
            probabilities(model,one,device,1)
            if device.type=="cuda":torch.cuda.synchronize()
            timings.append((perf_counter()-started)*1000)
        metrics["single_record_latency_ms"]={"samples":len(timings),
            "median":float(np.median(timings)),"p95":float(np.percentile(timings,95))}
        metadata={key:job[key] for key in ("id","model","dataset","task","classes","kind","name")}
        metadata.update({"model_version":config["experiment_id"]+"/"+job["id"]+"/"+job["kind"]+"/"+job["name"],
            "job_binding_sha256":canonical_hash(job_binding),"config":config,
            "pretrained_directory":str(pretrained),"pretrained_manifest_sha256":manifest_hash if job["model"]=="distilbert" else None,
            "preprocessing_variant":job.get("preprocessing_variant","control"),"test_evaluated":False,"positive_label":job["positive_label"]})
        if "heldout_domain" in job:
            metadata.update({key:job[key] for key in ("heldout_domain","training_domains","purged_training_rows")})
        save_final_model(model,directory,metadata,vocabulary)
        reloaded,reloaded_meta,_=load_final_model(directory,device=device)
        roundtrip=probabilities(reloaded,evaluation_encoded,device,spec["batch_size"])
        if not np.allclose(scores,roundtrip,rtol=1e-5,atol=1e-6):
            raise ValueError("Neural model reload changed probabilities")
        metrics["saved_model_reload_verified"]=True
        with (directory/"dev_predictions.csv").open("x",encoding="utf-8",newline="") as handle:
            writer=csv.writer(handle)
            writer.writerow(["record_id","partition","actual_label","predicted_label"]+
                            ["probability:"+label for label in job["classes"]])
            for identifier,label,probs in zip(job["evaluation"]["ids"],job["evaluation"]["labels"],scores):
                writer.writerow([identifier,"dev",label,job["classes"][int(np.argmax(probs))]]+probs.tolist())
        for filename,value in (("metrics.json",metrics),("history.json",trained["history"])):
            with (directory/filename).open("x",encoding="utf-8") as handle:
                json.dump(value,handle,indent=2,allow_nan=False)
                handle.write("\n")
        with (directory/"split_manifest.csv").open("xb") as handle:
            handle.write(manifest_bytes(job["manifest"]))
        files=["job.json","model.safetensors","metadata.json","dev_predictions.csv","metrics.json",
               "history.json","split_manifest.csv"]+(["vocabulary.json"] if vocabulary is not None else [])
        with (directory/"completion.json").open("x",encoding="utf-8") as handle:
            json.dump({"job_binding_sha256":canonical_hash(job_binding),
                "file_sha256":{filename:sha256(directory/filename) for filename in files},
                "completed_at_utc":datetime.now(timezone.utc).isoformat()},handle,indent=2)
            handle.write("\n")
        print(f"  Completed {job['id']}/{job['name']}: macro-F1={metrics['macro_f1']:.4f}",flush=True)
        del model,reloaded,train_encoded,evaluation_encoded
        if device.type=="cuda":torch.cuda.empty_cache()
    results=[]
    grouped={key:[] for key in pools}
    for job in jobs:
        directory=base/job["kind"]/job["id"]/job["name"]
        if not (directory/"completion.json").exists():
            continue
        job_binding={"run_binding_sha256":canonical_hash(binding),
                     "split_manifest_sha256":hashlib.sha256(manifest_bytes(job["manifest"])).hexdigest(),
                     "id":job["id"],"model":job["model"],"kind":job["kind"],
                     "name":job["name"],"classes":job["classes"],
                     "pretrained_manifest_sha256":manifest_hash if job["model"]=="distilbert" else None}
        result,predictions=read_complete(directory,job_binding,job)
        results.append(result)
        if job["kind"]=="generalization":grouped[job["id"]].append(predictions)
    aggregate=[]
    for key,pool in pools.items():
        expected=sum(job["id"]==key and job["kind"]=="generalization" for job in jobs)
        if len(grouped[key])==expected:
            metrics=aggregate_predictions(grouped[key],[r["record_id"] for r in pool["rows"]],
                                          pool["classes"],pool["positive_label"])
            aggregate.append({"id":key,"dataset":pool["dataset"],"task":pool["task"],
                              "model":pool["model"],"metrics":metrics})
    for filename,digest in protected.items():
        if sha256(ROOT/filename)!=digest:
            raise ValueError("Existing evidence changed: "+filename)
    summary={"schema_version":1,"completed_jobs":len(results),"declared_jobs":len(jobs),
        "complete":len(results)==len(jobs),"test_evaluated":False,"results":results,"aggregate":aggregate,
        "run_binding_sha256":canonical_hash(binding)}
    from chainshield.nlp.neural_train import atomic_json
    atomic_json(base/"summary.json",summary)
    report_path=ROOT/"docs/neural-experiments.md"
    text=report(results,aggregate,device)
    if not summary["complete"]:
        text="**STATUS: PARTIAL NEURAL RUN - "+str(len(results))+" of "+str(len(jobs))+" jobs complete.**\n\n"+text
    with report_path.open("w",encoding="utf-8") as handle:handle.write(text)
    print(f"Neural evidence saved: {len(results)}/{len(jobs)} jobs complete; final test untouched.",flush=True)


if __name__=="__main__":
    main()


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'configs/generalization.json': 'cb3512f6aad3a220dde4ca6fabce67ba7de46f53af85010f0e535cf1c0bd470f', 'src/chainshield/nlp/generalization.py': '7301ca723ab09a4f7c3528f48d1f0df188ccaef5ec197499638ceee72d9ca08e', 'scripts/evaluate_generalization.py': 'a8cd619f68345ac42c65fa0f5d787ec76732574e01d2a958aa79ae311a2c44fb', 'configs/neural.json': '5b8949f88a2add46a5704ed696236d6ba8cedc77110477181f311c02ad575d8b', 'src/chainshield/nlp/neural_train.py': '0f9226a1d7318c86143fdcd12d488274900f4cf69f1d2f68b00e54a152428006', 'scripts/train_neural.py': 'a653e5b0c534d2fe43836fc3feee2085f137e84d0dfa32f55b4802d44bfc1f31'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


## 5. Attach, verify prerequisites and execute

Bootstrap connects Drive and restores committed work. Source installation commits only this stage definitions. Dependencies must already have completed; this notebook does not silently train them.

In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:
        raise RuntimeError("Enable Drive/bootstrap first")
    rt.require_stage('04')
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
else:
    print("Stage execution disabled")


### Fit or reuse completed domain folds

This action executes only with RUN_STAGE=True. Progress and worker resource measurements are written to the experiment run directory.

In [ ]:
if RUN_STAGE:
    if not (rt.root/"artifacts/generalization/summary.json").exists():
        arguments=["scripts/evaluate_generalization.py"]
        if (rt.root/"artifacts/generalization/protocol.json").exists():
            arguments.append("--resume")
        rt.execute("06-generalization",arguments,SYNC_INTERVAL_SECONDS)
    else:
        print("Reusing twenty completed classical domain models")
else:
    print("Stage execution disabled")


## 6. Results and interpretation

These stress tests use original train/development pools; they are not untouched final-test estimates.

The following cell reads this experiment outputs; definitions alone are not evidence that a stage ran.

In [ ]:
if rt is None:
    print("No attached experiment")
else:
    path=rt.root/"artifacts/generalization/summary.json"
    if path.exists():
        for item in read_json(path)["aggregate"]:
            print(item["dataset"],item["task"],json.dumps(item["metrics"],indent=2))


## 7. Persist and hand over

Only after the preceding actions succeed, run the following cell to synchronize verified outputs and record this notebook completion. After an error, stop; repair/restart and rerun in order before recording completion. The next stage reads saved artifacts/components from Drive.

In [ ]:
if RUN_STAGE:
    rt.finish_stage('06')
else:
    print("Stage execution disabled")


## Recovery and boundaries

Run one writer per experiment. After interruption, disconnect and delete the Colab runtime, reconnect and restore this same experiment. Completed models are verified/reused; neural checkpoints resume compatible state. Unsynchronized work may repeat. Do not clear a writer lock until no other session is active.

No final-test fitting/scoring/tuning is performed. Model outputs do not verify incident truth or shipment exposure. Hosted Colab execution remains unverified until exercised in your session.